# 4-Hour Hands-on Tutorial: Artificial Neural Network Using Deep Learning

## Project: Customer Churn Prediction using ANN

You are working as a junior deep learning engineer for a telecom company. The business wants to predict whether a customer is likely to leave the company or continue using the service.

This Google Colab notebook is self-contained. It creates a synthetic telecom churn dataset, performs preprocessing, builds an ANN model, evaluates it, improves it using Dropout and EarlyStopping, saves the model, and predicts churn for a new customer.

## Learning Outcomes

By the end of this 4-hour tutorial, learners will be able to:

1. Explain ANN in simple terms.
2. Understand neurons, layers, weights, bias, activation functions, loss, optimizer, epochs, batch size, and learning rate.
3. Generate and understand a binary classification dataset.
4. Handle numerical and categorical data.
5. Build preprocessing pipelines.
6. Build, train, evaluate, and improve an ANN using TensorFlow/Keras.
7. Save and reuse a trained ANN model.
8. Make prediction for a new customer.

## 4-Hour Agenda

| Time | Module | Focus |
|---|---|---|
| 0:00 - 0:25 | Module 1 | ANN theory and business problem |
| 0:25 - 0:55 | Module 2 | Dataset creation and understanding |
| 0:55 - 1:25 | Module 3 | Exploratory data analysis |
| 1:25 - 1:55 | Module 4 | Data preprocessing |
| 1:55 - 2:35 | Module 5 | Build and train first ANN |
| 2:35 - 3:05 | Module 6 | Model evaluation |
| 3:05 - 3:35 | Module 7 | Improve ANN using Dropout and EarlyStopping |
| 3:35 - 3:50 | Module 8 | Save model and make new prediction |
| 3:50 - 4:00 | Module 9 | Assignment and viva questions |

# Module 1: ANN Concept in Layman Terms

ANN stands for **Artificial Neural Network**.

A neural network is made of small calculation units called **neurons**. These neurons are arranged in layers.

```text
Input Layer  →  Hidden Layers  →  Output Layer
```

## Customer Churn Example

Input data:

- Monthly charges
- Tenure
- Data usage
- Support calls
- Contract type
- Payment method

The ANN learns patterns like:

```text
High monthly charge + low tenure + many support calls = high churn chance
```

## Important Terms

| Term | Meaning |
|---|---|
| Neuron | Small calculation unit |
| Weight | Importance of input |
| Bias | Adjustment value |
| Hidden layer | Learns internal patterns |
| Activation function | Adds non-linearity |
| Epoch | One full pass over training data |
| Batch size | Number of records processed at once |
| Learning rate | Speed of learning |
| Loss | Model error |
| Optimizer | Updates weights to reduce error |

# Module 2: Environment Setup

Run this cell in Google Colab. TensorFlow is usually pre-installed, but this makes sure all required packages are available.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn tensorflow joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print("Libraries imported successfully")
print("TensorFlow version:", tf.__version__)

# Module 3: Create Synthetic Customer Churn Dataset

We will create a realistic telecom churn dataset.

## Target Variable

```text
Churn
```

| Value | Meaning |
|---|---|
| 0 | Customer will not churn |
| 1 | Customer will churn |

This is a **binary classification problem**.

In [ ]:
np.random.seed(42)

n = 5000

age = np.random.randint(18, 75, n)
tenure_months = np.random.randint(1, 73, n)
monthly_charges = np.round(np.random.normal(70, 25, n), 2)
monthly_charges = np.clip(monthly_charges, 20, 150)

data_usage_gb = np.round(np.random.gamma(shape=4, scale=12, size=n), 2)
support_calls = np.random.poisson(lam=2, size=n)
late_payments = np.random.poisson(lam=1, size=n)

contract_type = np.random.choice(
    ["Month-to-month", "One year", "Two year"],
    size=n,
    p=[0.55, 0.30, 0.15]
)

internet_service = np.random.choice(
    ["Fiber optic", "DSL", "No internet"],
    size=n,
    p=[0.50, 0.35, 0.15]
)

payment_method = np.random.choice(
    ["Electronic check", "Credit card", "Bank transfer", "Mailed check"],
    size=n,
    p=[0.40, 0.25, 0.25, 0.10]
)

gender = np.random.choice(["Male", "Female"], size=n)

# Business logic for churn probability
churn_score = (
    -0.035 * tenure_months
    + 0.025 * monthly_charges
    + 0.35 * support_calls
    + 0.40 * late_payments
    + 0.90 * (contract_type == "Month-to-month").astype(int)
    + 0.45 * (payment_method == "Electronic check").astype(int)
    + 0.35 * (internet_service == "Fiber optic").astype(int)
    - 2.8
)

churn_probability = 1 / (1 + np.exp(-churn_score))
churn = np.random.binomial(1, churn_probability)

df = pd.DataFrame({
    "Age": age,
    "Tenure_Months": tenure_months,
    "Monthly_Charges": monthly_charges,
    "Data_Usage_GB": data_usage_gb,
    "Support_Calls": support_calls,
    "Late_Payments": late_payments,
    "Contract_Type": contract_type,
    "Internet_Service": internet_service,
    "Payment_Method": payment_method,
    "Gender": gender,
    "Churn": churn
})

df.head()

## Add Missing Values

Real-world data often contains missing values. We intentionally add missing values to selected columns so that students can practice missing value treatment.

In [ ]:
missing_columns = ["Monthly_Charges", "Data_Usage_GB", "Contract_Type", "Payment_Method"]

for col in missing_columns:
    missing_indices = np.random.choice(df.index, size=int(0.03 * n), replace=False)
    df.loc[missing_indices, col] = np.nan

df.isnull().sum()

# Student Task 1

Answer:

1. What is the target variable?
2. Is this classification or regression?
3. Which columns are numerical?
4. Which columns are categorical?
5. Why can churn prediction help a telecom company?

# Module 4: Exploratory Data Analysis

EDA means understanding the dataset before building the model.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True) * 100

In [ ]:
plt.figure(figsize=(6, 4))
df["Churn"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.title("Churn Distribution: 0 = Not Churn, 1 = Churn")
plt.xticks(rotation=0)
plt.show()

In [ ]:
for col in ["Age", "Tenure_Months", "Monthly_Charges", "Data_Usage_GB", "Support_Calls"]:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col].dropna(), bins=30)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
contract_churn = df.groupby("Contract_Type")["Churn"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
contract_churn.plot(kind="bar")
plt.ylabel("Average Churn Rate")
plt.title("Churn Rate by Contract Type")
plt.xticks(rotation=30)
plt.show()

contract_churn

In [ ]:
payment_churn = df.groupby("Payment_Method")["Churn"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
payment_churn.plot(kind="bar")
plt.ylabel("Average Churn Rate")
plt.title("Churn Rate by Payment Method")
plt.xticks(rotation=30)
plt.show()

payment_churn

# Module 5: Data Preparation for ANN

ANN models need numerical input. So we will:

1. Separate features and target.
2. Split data into training and testing sets.
3. Fill missing numerical values.
4. Scale numerical values.
5. Fill missing categorical values.
6. One-hot encode categorical values.

In [ ]:
target = "Churn"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

## Build Preprocessing Pipeline

For ANN, scaling numerical features is important because neural networks learn better when input values are on a similar scale.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created")

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

# Student Task 2

Answer:

1. Why do we use train-test split?
2. Why do we use stratify in classification?
3. Why do we scale numerical columns before ANN?
4. Why do we one-hot encode categorical columns?
5. Why do we fit the preprocessor only on training data?

# Module 6: Build First ANN Model

Architecture:

```text
Input Layer
    ↓
Dense Layer with ReLU
    ↓
Dense Layer with ReLU
    ↓
Output Layer with Sigmoid
```

Since churn prediction is binary classification, the output layer uses **sigmoid** activation.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

input_dim = X_train_processed.shape[1]

ann_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()

## ANN Code Explanation

| Code | Meaning |
|---|---|
| `Input(shape=(input_dim,))` | Number of input features after preprocessing |
| `Dense(64, activation="relu")` | Hidden layer with 64 neurons |
| `Dense(32, activation="relu")` | Hidden layer with 32 neurons |
| `Dense(1, activation="sigmoid")` | Output layer for binary classification |
| `binary_crossentropy` | Loss function for binary classification |
| `Adam` | Optimizer to update weights |
| `learning_rate=0.001` | Speed of learning |

In [ ]:
history = ann_model.fit(
    X_train_processed,
    y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# Module 7: Evaluate ANN Model

We will evaluate the model using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix
- Classification Report
- ROC Curve

In [ ]:
y_pred_proba = ann_model.predict(X_test_processed).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("ANN Model Evaluation")
print("--------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["Not Churn", "Churn"])
plt.yticks([0, 1], ["Not Churn", "Churn"])
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Not Churn", "Churn"]))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ANN ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Module 8: Improve ANN using Dropout and EarlyStopping

## Overfitting

Overfitting means the model performs well on training data but poorly on unseen testing data.

## Dropout

Dropout randomly switches off some neurons during training.

Layman meaning:

```text
The model should not depend too much on only a few neurons.
```

## EarlyStopping

EarlyStopping stops training when validation loss stops improving.

Layman meaning:

```text
Stop training before the model starts memorizing the data.
```

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

improved_ann_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation="relu"),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.20),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

improved_ann_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

improved_ann_model.summary()

In [ ]:
improved_history = improved_ann_model.fit(
    X_train_processed,
    y_train,
    validation_split=0.20,
    epochs=80,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(improved_history.history["accuracy"], label="Training Accuracy")
plt.plot(improved_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Improved ANN: Training vs Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(improved_history.history["loss"], label="Training Loss")
plt.plot(improved_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Improved ANN: Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
improved_pred_proba = improved_ann_model.predict(X_test_processed).ravel()
improved_pred = (improved_pred_proba >= 0.5).astype(int)

improved_accuracy = accuracy_score(y_test, improved_pred)
improved_precision = precision_score(y_test, improved_pred)
improved_recall = recall_score(y_test, improved_pred)
improved_f1 = f1_score(y_test, improved_pred)
improved_roc_auc = roc_auc_score(y_test, improved_pred_proba)

print("Improved ANN Model Evaluation")
print("-----------------------------")
print("Accuracy:", round(improved_accuracy, 4))
print("Precision:", round(improved_precision, 4))
print("Recall:", round(improved_recall, 4))
print("F1 Score:", round(improved_f1, 4))
print("ROC-AUC:", round(improved_roc_auc, 4))

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Simple ANN",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    },
    {
        "Model": "Improved ANN",
        "Accuracy": improved_accuracy,
        "Precision": improved_precision,
        "Recall": improved_recall,
        "F1 Score": improved_f1,
        "ROC-AUC": improved_roc_auc
    }
])

comparison_df

# Module 9: Threshold Tuning

By default, a customer is classified as churn if probability is greater than or equal to 0.5.

Businesses can choose a different threshold.

- Lower threshold catches more churn customers.
- Higher threshold gives fewer but more confident churn predictions.

In [ ]:
threshold_results = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_threshold = (improved_pred_proba >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred_threshold),
        "Precision": precision_score(y_test, pred_threshold),
        "Recall": recall_score(y_test, pred_threshold),
        "F1 Score": f1_score(y_test, pred_threshold)
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

# Module 10: Save and Reuse ANN Model

We save:

1. The preprocessing pipeline
2. The trained ANN model

Why both?

The model expects processed/scaled/encoded data. So every new customer must go through the same preprocessor.

In [ ]:
joblib.dump(preprocessor, "ann_churn_preprocessor.pkl")
improved_ann_model.save("ann_churn_model.keras")

print("Preprocessor and ANN model saved successfully")

In [ ]:
loaded_preprocessor = joblib.load("ann_churn_preprocessor.pkl")
loaded_model = tf.keras.models.load_model("ann_churn_model.keras")

print("Model and preprocessor loaded successfully")

# Module 11: New Customer Prediction

In [ ]:
new_customer = pd.DataFrame({
    "Age": [35],
    "Tenure_Months": [6],
    "Monthly_Charges": [105.50],
    "Data_Usage_GB": [85.20],
    "Support_Calls": [5],
    "Late_Payments": [3],
    "Contract_Type": ["Month-to-month"],
    "Internet_Service": ["Fiber optic"],
    "Payment_Method": ["Electronic check"],
    "Gender": ["Female"]
})

new_customer

In [ ]:
new_customer_processed = loaded_preprocessor.transform(new_customer)

if hasattr(new_customer_processed, "toarray"):
    new_customer_processed = new_customer_processed.toarray()

new_customer_probability = loaded_model.predict(new_customer_processed).ravel()[0]
new_customer_prediction = 1 if new_customer_probability >= 0.5 else 0

print("Churn Probability:", round(new_customer_probability, 4))
print("Predicted Class:", new_customer_prediction)

if new_customer_prediction == 1:
    print("Business Interpretation: Customer is likely to churn.")
else:
    print("Business Interpretation: Customer is not likely to churn.")

# Module 12: Student Assignment

## Assignment 1: Change ANN Architecture

Try these architectures:

1. 32 → 16 → 1
2. 128 → 64 → 32 → 1
3. 256 → 128 → 64 → 1

Record:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

## Assignment 2: Change Dropout

Try dropout values:

```text
0.1, 0.2, 0.3, 0.5
```

Observe whether overfitting reduces.

## Assignment 3: Threshold Selection

Try thresholds:

```text
0.25, 0.35, 0.45, 0.55, 0.65
```

Which threshold gives the best precision-recall balance?

## Assignment 4: Business Interpretation

Write a short business explanation:

1. Which customers are more likely to churn?
2. Which metric is most important for churn prediction?
3. How can the telecom company use this model?

# Viva Questions

1. What is an Artificial Neural Network?
2. What is a neuron?
3. What is the role of weights and bias?
4. Why do we use activation functions?
5. Why is ReLU used in hidden layers?
6. Why is sigmoid used in the output layer?
7. What is binary crossentropy?
8. What is an optimizer?
9. What is learning rate?
10. What is an epoch?
11. What is batch size?
12. What is overfitting?
13. How does Dropout reduce overfitting?
14. What is EarlyStopping?
15. Why do we scale numerical features before ANN?
16. What is ROC-AUC?
17. Why is recall important in churn prediction?
18. Why do we save the preprocessor along with the ANN model?

# Final Summary

In this 4-hour hands-on tutorial, we built an end-to-end ANN project for customer churn prediction.

We learned:

```text
Raw data
→ EDA
→ Missing value handling
→ Scaling and encoding
→ ANN model building
→ Model training
→ Model evaluation
→ Dropout and EarlyStopping
→ Model saving
→ New customer prediction
```

Key takeaway:

> ANN is a deep learning model that learns patterns using neurons, weights, bias, activation functions, loss function, and optimizer. For tabular classification problems like churn prediction, ANN can learn useful non-linear relationships from customer data.